In [1]:
!apt-get install tesseract-ocr
!pip install pytesseract pillow

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 6 not upgraded.


In [9]:
from google.colab import files

uploaded = files.upload()

Saving Screenshot 2026-03-27 150703.png to Screenshot 2026-03-27 150703.png


In [10]:
import pytesseract
from PIL import Image
import re
import io

def clean_text(text):
    text = re.sub(r'[^A-Za-z0-9.\n ]', '', text)
    text = re.sub(r'\n+', '\n', text)
    return text.strip()

def extract_numbers(text):
    return re.findall(r'\d+\.\d{2}|\d+', text)

for filename in uploaded.keys():

    image = Image.open(io.BytesIO(uploaded[filename]))

    text = pytesseract.image_to_string(image)

    cleaned_text = clean_text(text)

    numbers = extract_numbers(cleaned_text)

    print("FULL TEXT\n")
    print(cleaned_text)

    print("\nNUMBERS FOUND\n")
    for n in numbers:
        print(n)

    if numbers:
        total = float(numbers[-1])
        print("\nTOTAL:", total)

        if total > 500:
            print("High expense")
        else:
            print("Normal expense")

FULL TEXT

SUPERMARKET
Lorem ipsum 288
City Index  02025
Tel 45646898702
 
Cashier 3
Manager Eric Steer
 
Name ay Price
Lorem ipsum 1 9.20
Lorem ipsum dolor sit 1 19.20
Lorem ipsum dolor sit amet 115.00
Lorem ipsum 1 15.00
Lorem ipsum 1 18.00
Lorem ipsum dolor sit 1 15.00
Lorem ipsum 1 1920
 
Sub Total 107.60
CASH 200.00
CHANGE 92.40
THANK YOU

NUMBERS FOUND

288
02025
45646898702
3
1
9.20
1
19.20
115.00
1
15.00
1
18.00
1
15.00
1
1920
107.60
200.00
92.40

TOTAL: 92.4
Normal expense


## Was It Easier to Use?

Yes.
The extracted text became cleaner and easier to read
Numbers were easier to identify
Output looked closer to the original receipt format
The code itself became simpler and more readable

The improvement makes the OCR tool more practical for analyzing receipts, since noisy characters from OCR are filtered out.

## Where does OCR stop and “understanding” begin?

OCR stops at reading the text from the image. It only converts what it sees in the picture into machine-readable text.

Understanding begins when the program interprets that text and makes decisions from it. For example, after OCR extracts numbers from a receipt, the program then decides which number might be the total or whether the expense is high or normal.

So OCR = reading text
Understanding = interpreting the meaning of that text.


## Did the system actually understand the receipt?

Not really.The system only guessed that the last number in the receipt might be the total. It did not truly understand the structure of a receipt or know which value represents the total amount.

What would be needed to make it smarter?

To make the system smarter, we would need:

1. Better text processing
Search for keywords like Total, Amount, Grand Total instead of just selecting the last number.

2. Layout analysis
Understand where text appears on the receipt so it can detect sections like item lists and totals.

3. Machine learning models
Train models to recognize receipt structure.

4. NLP (Natural Language Processing)
Use language models to interpret text and extract useful fields like date, store name, and total amount.

5. More training data
Use many examples of receipts so the system learns common patterns.

In [11]:
from google.colab import files
import pytesseract
from PIL import Image
import re
import io

uploaded = files.upload()

def clean_text(text):
    text = re.sub(r'[^A-Za-z0-9.\n ]', '', text)
    text = re.sub(r'\n+', '\n', text)
    return text.strip()

def is_receipt(text):
    keywords = ["total", "tax", "subtotal", "amount"]
    text = text.lower()
    for k in keywords:
        if k in text:
            return True
    return False

def extract_total(text):
    match = re.search(r'(total\s*[: ]?\s*)(\d+\.\d{2})', text.lower())
    if match:
        return match.group(2)
    return None

def extract_date(text):
    match = re.search(r'\d{2}/\d{2}/\d{4}', text)
    if match:
        return match.group()
    return None


for filename in uploaded.keys():

    print("\nProcessing file:", filename)

    image = Image.open(io.BytesIO(uploaded[filename]))

    text = pytesseract.image_to_string(image)

    cleaned_text = clean_text(text)

    if is_receipt(cleaned_text):
        print("Receipt detected")
    else:
        print("This image might not be a receipt")

    print("\nExtracted Text:\n")
    print(cleaned_text)

    total = extract_total(cleaned_text)
    date = extract_date(cleaned_text)

    print("\nDate:", date)
    print("Total:", total)

    numbers = re.findall(r'\d+\.\d{2}', cleaned_text)

    print("\nNumbers found in receipt:")
    for n in numbers:
        print(n)

Saving Screenshot 2026-03-27 144502.png to Screenshot 2026-03-27 144502 (1).png

Processing file: Screenshot 2026-03-27 144502 (1).png
Receipt detected

Extracted Text:

Sdertalje Weda
Org 5561632232
Tn 08553 776 50
 
MJOLK 3 LANGR HALLB 5st1450 7250
5  RabattMJOLK 1165
AYRAN YOGHURTDRYCK  2st 1850 3700
CHOKLADKROSS GLASS 4290
KOLSYRAT VATTEN CITR 990
PANT ENG PET 1L 200
PREMIER MANGO 151 990
2  RabattDRYCK 300
PANT ENG PET 1L 200
HET KEBABSAS 2990
MILD KEBABSAS 2990
NORMALSALT SMORRAP 5480
MELLANSALT500G SMRA 5380
2  Willys PlusBREGOTT 3332
KVARG VANILJ 02 4290
UPGO 7 1626KG 24P 9890
RabattBLOJOR 900
GUL LOK 1190
RabattGUL LOK 250
TORTILLA CHIPS CHEES 1350
LANTBROD HAVSSALT 3290
Willys PlusLANTBROD 941
DUOKEX CHOKLAD 1490
HAVREKEX HOBNOBS 1590
 
Totalt 45662 SEK

Date: None
Total: None

Numbers found in receipt:
